In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
# Read in all the words
words = open('names.txt', 'r').read().splitlines()

In [4]:
# Build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [50]:
 # Build the dataset

block_size = 3 # context length: How many characters do we take to predict the next one?
X, Y = [], []
for w in words:
    #print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context), '----->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)


In [51]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [127]:
 # Build the dataset with train, validate, test splits 80/10/10

def build_dataset(words):
    block_size = 3 # context length: How many characters do we take to predict the next one?
    X, Y = [], []
    for w in words:
        #print(w)
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            #print(''.join(itos[i] for i in context), '----->', itos[ix])
            context = context[1:] + [ix] # crop and append

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])


torch.Size([182580, 3]) torch.Size([182580])
torch.Size([22767, 3]) torch.Size([22767])
torch.Size([22799, 3]) torch.Size([22799])


In [8]:
C = torch.randn(27, 2) 
# C is a lookup table with 27 rows and 2 columns, where 27 represents chars, and 2 represents features for each char
 

In [10]:
# C[5]

tensor([-0.4959, -0.0090])

In [12]:
# F.one_hot(torch.tensor(5), num_classes = 27).float() @ C


tensor([-0.4959, -0.0090])

In [13]:
# C[X].shape

torch.Size([32, 3, 2])

In [17]:
# X[13, 2] # 13 is . . a , above, so [13, 2] will be a which have index 1

tensor(1)

In [21]:
# C[X][13,2] # Lookup for 13,2 which is 1 gives the same as lookup for 1

tensor([-0.0041,  0.1393])

In [22]:
# C[1] # Lookup for 1 gives the same as lookup for 13, 2

tensor([-0.0041,  0.1393])

C - Embedding Matrix - (27, 2) each character has 2 numbers (features)


X - Input indices - (N, 3) N is the no. of Combos with 3 letter context and 3 is for number of letters in context


emb - Concatenated Embeddings - ([N, 3, 2]) Pulling out the rows from C that match the IDs in X


w1 - Layer 1 Weights - (6,100) 6 matches the total size of flattenedinput embedding and 100 represents the number of neurons in hidden layer


b1 - Layer 1 Biases - (100) 100 mtches the hidden neurons 

In [23]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [24]:
w1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [25]:
# torch.cat(torch.unbind(emb, 1), 1).shape One way to change shape

torch.Size([32, 6])

a.view() is efficient compared to torch.cat(torch.unbind()), because for view() the underlying storage (a.storage()) doesn't change , but it changes for other things

In [26]:
# -1 represents a wildcard that tells the library to automatically calculate the specific dimension for you
h = torch.tanh(emb.view(-1, 6) @ w1 + b1) 

In [28]:
w2 = torch.randn((100,27))
b2 = torch.randn(27)

In [29]:
logits = h @ w2 + b2

In [30]:
logits.shape

torch.Size([32, 27])

In [31]:
counts = logits.exp()

In [36]:
probs = counts / counts.sum(1, keepdim=True)

In [37]:
probs.shape

torch.Size([32, 27])

In [38]:
torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [39]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [41]:
loss = -probs[torch.arange(32), Y].log().mean()
loss

tensor(41.3107)

Clean Code

Always use F.cross_entropy(), dont do manually (counts,probs, loss) in production. Beacause , if numbers for high negative or high positive , the probs goes either zero or nan. where as pytorch first subtracts the maximum number from the whole logits which avoids that zero or nan bugs

# counts = logits.exp()
# probs = counts / counts.sum(1, keepdim=True)
# loss = -probs[torch.arange(32), Y].log().mean()

In [130]:
X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [136]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator = g)
w1 = torch.randn((6, 300), generator = g)
b1 = torch.randn(300, generator = g)
w2 = torch.randn((300, 27), generator = g)
b2 = torch.randn(27, generator = g)
parameters = [C, w1, b1, w2, b2]


In [137]:
sum(p.nelement() for p in parameters)

10281

In [133]:
for p in parameters:
    p.requires_grad = True

In [104]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [119]:
lri = []
lossi = []
for i in range(10000):
    # Minibtach Construct
    ix = torch.randint(0, X.shape[0], (32,))

    # forward pass
    emb = C[X[ix]] #(32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ w1 + b1) #(32, 100)
    logits = h @ w2 + b2 #(32, 27)
    loss = F.cross_entropy(logits, Y[ix])

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    
    # Update
    # lr = lrs[i]
    lr = 0.01 # Initially start with 0.1 and as we come to an end of train , use learning decay(reduce the lr by a factor of 10)
    for p in parameters:
        p.data += -lr * p.grad

    # Track Stats
    # lri.append(lr)
    # lossi.append(loss.item())

print("Loss of mini batch size 32:",loss.item())


Loss of mini batch size 32: 2.190302848815918


In [124]:
# plt.plot(lri, lossi) # This is the how we determine the optimal learning rate for a model, which in this case is 0.1

In [123]:
emb = C[X] #(32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ w1 + b1) #(32, 100)
logits = h @ w2 + b2 #(32, 27)
loss = F.cross_entropy(logits, Y)
print("Loss of whole data:",loss.item())

Loss of whole data: 2.3094756603240967


# training split, dev/validation split, test split
#80% /10% /10%
# Training is to train the parameters
# DEv/ validation is to train the hyper parameters(hidden layers)
# test is used to evaluate the performance of model at the end

In [215]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 10), generator = g)
w1 = torch.randn((30, 200), generator = g)
b1 = torch.randn(200, generator = g)
w2 = torch.randn((200, 27), generator = g)
b2 = torch.randn(27, generator = g)
parameters = [C, w1, b1, w2, b2]

In [216]:
sum(p.nelement() for p in parameters)

11897

In [217]:
for p in parameters:
    p.requires_grad = True

In [218]:
lri = []
lossi = []
stepi = []


In [219]:
for i in range(200000):
    # Minibtach Construct
    ix = torch.randint(0, Xtr.shape[0], (32,))

    # forward pass
    emb = C[Xtr[ix]] #(32, 3, 2)
    h = torch.tanh(emb.view(-1, 30) @ w1 + b1) #(32, 100)
    logits = h @ w2 + b2 #(32, 27)
    loss = F.cross_entropy(logits, Y[ix])

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    
    # Update
    # lr = lrs[i]
    lr = 0.1 if i<100000 else 0.05 # Initially start with 0.1 and as we come to an end of train , use learning decay(reduce the lr by a factor of 10)
    for p in parameters:
        p.data += -lr * p.grad

    # Track Stats
    # lri.append(lr)
    stepi.append(i)
    lossi.append(loss.log10().item())

print("Loss of mini batch size 32:",loss.item())


Loss of mini batch size 32: 2.9085335731506348


In [201]:
#plt.plot(stepi, lossi)

In [205]:
emb = C[Xtr] #(32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ w1 + b1) #(32, 100)
logits = h @ w2 + b2 #(32, 27)
loss = F.cross_entropy(logits, Ytr)
print("Loss of training split data:",loss.item())

Loss of training split data: 2.8646514415740967


In [206]:
emb = C[Xdev] 
h = torch.tanh(emb.view(-1, 30) @ w1 + b1) 
logits = h @ w2 + b2 #(32, 27)
loss = F.cross_entropy(logits, Ydev)
print("Loss of Dev split data:",loss.item())

Loss of Dev split data: 2.8588650226593018


In [159]:
#plt.plot(stepi, lossi)

In [214]:
# Sample from the model
g = torch.Generator().manual_seed(2147483647)

for _ in range(20):

    out = []
    context =[0] * block_size
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ w1 + b1)
        logits = h @ w2 + b2
        probs = F.softmax( logits, dim =1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        out.append(ix)
        if ix==0:
            break

    print(''.join(itos[i] for i in out))

cexzm.
zoglkurkicczktyhwbvmzimjttainrlkfukzkktda.
sfcxvpubjtbhrmgotzx.
iczixqckvugkwptedogkkjemkmmsidguenkbvgynywftbspmhwcivgbvtahlvsu.
dsdxxblnwglhpyiw.
igwnjwrpfdwipkwzkm.
desu.
firmt.
gbiksjbquabsvoth.
kuysxqevhcmrbxmcwyhrrjenvxmvpfkiwmghfvjzxobomysox.
gbptjapxweegpfwhccfyzfvksiiqmvwbhmiwqmdgzqskmjhgamcxwmckciswcxfmbalcslhy.
fpycdasvz.
bqzazeunschck.
wnkojuoxyvtvfiwksddugnkul.
fuwycgjz.
abl.
j.
nuuutstofgqzubbo.
rdubpknumd.
vhfacdvaadsjzjkdh.
